In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

In [2]:
# Paths
repo_root = Path.cwd()
if not (repo_root / "results").exists() and (repo_root.parent / "results").exists():
    repo_root = repo_root.parent

RESULTS_DIR = repo_root / "results" / "arch_grid_search_shards"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found: {RESULTS_DIR}")

print("Using repo_root:", repo_root)
print("Using results dir:", RESULTS_DIR)

Using repo_root: /ocean/projects/cis260122p/gfeng/lob-deep-survival-analysis
Using results dir: /ocean/projects/cis260122p/gfeng/lob-deep-survival-analysis/results/arch_grid_search_shards


In [26]:
# Discover shard result files
csv_files = sorted(RESULTS_DIR.glob("arch_grid_results_*.csv"))
json_files = sorted(RESULTS_DIR.glob("arch_grid_results_*.json"))

print(f"Found {len(csv_files)} shard CSV files")
print(f"Found {len(json_files)} shard metadata files")

if len(csv_files) == 0:
    raise ValueError(f"No shard CSV files found in {RESULTS_DIR}")

Found 1148 shard CSV files
Found 1148 shard metadata files


In [27]:
# ---- Optional filtering by array_job_id ----

# Set to None to load everything
# Or provide a list like: ["123456", "123457"]
FILTER_JOB_IDS = ['40166470']

def extract_job_id_from_filename(path: Path) -> str | None:
    """
    Extract array_job_id from filename pattern:
    arch_grid_results_<config>_<jobid>_<taskid>.csv
    """
    parts = path.stem.split("_")
    if len(parts) < 2:
        return None
    # last two parts are jobid and taskid
    # e.g. ..._<jobid>_<taskid>
    try:
        job_id = parts[-2]
        return job_id
    except Exception:
        return None


filtered_csv_files = []

for path in csv_files:
    job_id = extract_job_id_from_filename(path)

    if FILTER_JOB_IDS is None:
        keep = True
    else:
        keep = job_id in set(str(x) for x in FILTER_JOB_IDS)

    if keep:
        filtered_csv_files.append(path)

print(f"CSV files before filtering: {len(csv_files)}")
print(f"CSV files after filtering: {len(filtered_csv_files)}")

if len(filtered_csv_files) == 0:
    raise ValueError("No CSV files matched FILTER_JOB_IDS")

# overwrite csv_files for downstream cells
csv_files = sorted(filtered_csv_files)

CSV files before filtering: 1148
CSV files after filtering: 48


In [28]:
def safe_read_csv(path: Path) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(path)
        df["source_csv"] = path.name
        return df
    except Exception as e:
        warnings.warn(f"Failed to read {path.name}: {e}")
        return None

dfs = []
for path in csv_files:
    df = safe_read_csv(path)
    if df is not None and len(df) > 0:
        dfs.append(df)

if not dfs:
    raise ValueError("All shard CSV files failed to load or were empty.")

raw_df = pd.concat(dfs, ignore_index=True)
print("Merged raw rows:", len(raw_df))
print("Columns:", sorted(raw_df.columns.tolist()))

Merged raw rows: 48
Columns: ['alpha', 'array_job_id', 'array_task_id', 'beta_l3', 'config_id', 'created_at', 'depth_repr', 'fc_dropout', 'fc_hidden_ratio', 'gru_layers', 'hidden_size', 'learning_rate', 'model_name', 'num_epochs', 'rnn_dropout', 'run_name', 'seed', 'selection_max_params', 'selection_min_params', 'source_csv', 'ticker', 'trainable_params', 'trainable_params_m', 'transformer_dropout', 'transformer_ff_ratio', 'transformer_heads', 'transformer_layers', 'val_ctd_favorable', 'val_ctd_toxic', 'val_ibs_favorable', 'val_ibs_toxic', 'val_ibs_weighted', 'val_ibs_weighted_uninformed', 'val_toxic_pr_auc_sample', 'val_toxic_pr_auc_weighted', 'val_toxic_roc_auc_sample', 'val_toxic_roc_auc_weighted', 'val_weighted_ctd']


In [29]:
# Basic normalization / numeric conversion
metric_cols = [
    "val_weighted_ctd",
    "val_toxic_pr_auc_weighted",
    "val_toxic_pr_auc_sample",
    "val_ctd_toxic",
    "val_ctd_favorable",
    "val_ibs_weighted",
]

for col in metric_cols:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Normalize key ID fields
for col in ["config_id", "model_name", "ticker", "source_csv"]:
    if col in raw_df.columns:
        raw_df[col] = raw_df[col].astype(str)

if "seed" in raw_df.columns:
    raw_df["seed"] = pd.to_numeric(raw_df["seed"], errors="coerce").astype("Int64")

print(raw_df[["config_id", "model_name", "ticker", "seed"] + [c for c in metric_cols if c in raw_df.columns]].head())

      config_id       model_name ticker  seed  val_weighted_ctd  \
0  25b12b8d778d  gru_transformer   AAPL  4718          0.607299   
1  25b12b8d778d  gru_transformer   AAPL  4819          0.571544   
2  25b12b8d778d  gru_transformer   INTC  4718          0.778338   
3  25b12b8d778d  gru_transformer   INTC  4819          0.789459   
4  25b12b8d778d  gru_transformer   LCID  4718          0.900549   

   val_toxic_pr_auc_weighted  val_toxic_pr_auc_sample  val_ctd_toxic  \
0                   0.265046                 0.209606       0.695124   
1                   0.320076                 0.246973       0.638603   
2                   0.638268                 0.486293       0.810016   
3                   0.663469                 0.517553       0.815813   
4                   0.562008                 0.433817       0.933132   

   val_ctd_favorable  val_ibs_weighted  
0           0.587676          0.235499  
1           0.556561          0.222164  
2           0.754916          0.241086  


In [30]:
# Error handling for failed shards / incomplete rows:
# Keep rows even if some metrics are missing, but exclude missing metrics from each ranking calculation.

required_id_cols = ["config_id", "model_name", "ticker", "seed"]
missing_id_cols = [c for c in required_id_cols if c not in raw_df.columns]
if missing_id_cols:
    raise ValueError(f"Missing required ID columns: {missing_id_cols}")

# Detect duplicate (config, ticker, seed) rows and keep the best available one
# Preference order:
# 1) row with val_weighted_ctd present
# 2) row with val_toxic_pr_auc_weighted present
# 3) first row otherwise
dedup_df = raw_df.copy()
dedup_df["_has_ctd"] = dedup_df["val_weighted_ctd"].notna() if "val_weighted_ctd" in dedup_df.columns else False
dedup_df["_has_prauc"] = dedup_df["val_toxic_pr_auc_weighted"].notna() if "val_toxic_pr_auc_weighted" in dedup_df.columns else False

dedup_df = dedup_df.sort_values(
    by=["config_id", "ticker", "seed", "_has_ctd", "_has_prauc"],
    ascending=[True, True, True, False, False],
)

dedup_df = dedup_df.drop_duplicates(subset=["config_id", "ticker", "seed"], keep="first").copy()

print("Rows after dedup:", len(dedup_df))
print("Unique configs:", dedup_df["config_id"].nunique())

Rows after dedup: 48
Unique configs: 8


In [31]:
# Coverage diagnostics
coverage_df = (
    dedup_df.groupby(["config_id", "model_name"], as_index=False)
    .agg(
        n_rows=("config_id", "size"),
        n_unique_tickers=("ticker", "nunique"),
        n_unique_seeds=("seed", "nunique"),
        n_ctd=("val_weighted_ctd", lambda s: s.notna().sum()),
        n_prauc=("val_toxic_pr_auc_weighted", lambda s: s.notna().sum()),
        trainable_params=("trainable_params", "first") if "trainable_params" in dedup_df.columns else ("config_id", "size"),
        hidden_size=("hidden_size", "first") if "hidden_size" in dedup_df.columns else ("config_id", "size"),
    )
    .sort_values(["model_name", "config_id"])
    .reset_index(drop=True)
)

coverage_df["expected_rows"] = 6  # 3 tickers x 2 seeds
coverage_df["missing_rows"] = coverage_df["expected_rows"] - coverage_df["n_rows"]

display(coverage_df.head(20))

,config_id,model_name,n_rows,n_unique_tickers,n_unique_seeds,n_ctd,n_prauc,trainable_params,hidden_size,expected_rows,missing_rows
0,25b12b8d778d,gru_transformer,6,3,2,6,6,304365,96,6,0
1,3faa61ed2ce7,gru_transformer,6,3,2,6,6,266637,96,6,0
2,55bdbf1bed26,gru_transformer,6,3,2,6,6,266637,96,6,0
3,9a1d0a1f723a,gru_transformer,6,3,2,6,6,397293,96,6,0
4,ab7ab945565f,gru_transformer,6,3,2,6,6,248109,96,6,0
5,bcc3408502b0,gru_transformer,6,3,2,6,6,229581,96,6,0
6,c2b98c4f4e50,gru_transformer,6,3,2,6,6,248109,96,6,0
7,d217626fd507,gru_transformer,6,3,2,6,6,285453,96,6,0


In [32]:
# Aggregate metrics per config, using available rows only
agg_dict = {
    "val_weighted_ctd": "mean",
    "val_toxic_pr_auc_weighted": "mean",
}

extra_first_cols = [
    "trainable_params",
    "trainable_params_m",
    "hidden_size",
    "gru_layers",
    "transformer_layers",
    "transformer_heads",
    "transformer_ff_ratio",
    "mamba_layers",
    "mamba_d_state",
    "mamba_d_conv",
    "mamba_expand",
    "depth_repr",
]

group_cols = ["config_id", "model_name"]

agg_spec = {
    "avg_weighted_ctd": ("val_weighted_ctd", "mean"),
    "avg_toxic_pr_auc": ("val_toxic_pr_auc_weighted", "mean"),
    "n_ctd": ("val_weighted_ctd", lambda s: s.notna().sum()),
    "n_prauc": ("val_toxic_pr_auc_weighted", lambda s: s.notna().sum()),
    "n_total_rows": ("config_id", "size"),
    "n_unique_tickers": ("ticker", "nunique"),
    "n_unique_seeds": ("seed", "nunique"),
}

for col in extra_first_cols:
    if col in dedup_df.columns:
        agg_spec[col] = (col, "first")

rank_df = (
    dedup_df.groupby(group_cols, as_index=False)
    .agg(**agg_spec)
    .copy()
)

rank_df["expected_rows"] = 6
rank_df["ctd_complete"] = rank_df["n_ctd"] == rank_df["expected_rows"]
rank_df["prauc_complete"] = rank_df["n_prauc"] == rank_df["expected_rows"]

display(rank_df.head(20))

,config_id,model_name,avg_weighted_ctd,avg_toxic_pr_auc,n_ctd,n_prauc,n_total_rows,n_unique_tickers,n_unique_seeds,trainable_params,trainable_params_m,hidden_size,gru_layers,transformer_layers,transformer_heads,transformer_ff_ratio,depth_repr,expected_rows,ctd_complete,prauc_complete
0,25b12b8d778d,gru_transformer,0.763303,0.491480,6,6,6,3,2,304365,0.304365,96,1.0,2.0,4.0,2.0,"gru=1.0,tr=2.0",6,True,True
1,3faa61ed2ce7,gru_transformer,0.773341,0.509871,6,6,6,3,2,266637,0.266637,96,1.0,1.0,4.0,4.0,"gru=1.0,tr=1.0",6,True,True
2,55bdbf1bed26,gru_transformer,0.764878,0.506592,6,6,6,3,2,266637,0.266637,96,1.0,1.0,2.0,4.0,"gru=1.0,tr=1.0",6,True,True
3,9a1d0a1f723a,gru_transformer,0.764895,0.478682,6,6,6,3,2,397293,0.397293,96,2.0,2.0,4.0,3.0,"gru=2.0,tr=2.0",6,True,True
4,ab7ab945565f,gru_transformer,0.774783,0.511980,6,6,6,3,2,248109,0.248109,96,1.0,1.0,2.0,3.0,"gru=1.0,tr=1.0",6,True,True
5,bcc3408502b0,gru_transformer,0.751563,0.511499,6,6,6,3,2,229581,0.229581,96,1.0,1.0,4.0,2.0,"gru=1.0,tr=1.0",6,True,True
6,c2b98c4f4e50,gru_transformer,0.763838,0.509056,6,6,6,3,2,248109,0.248109,96,1.0,1.0,4.0,3.0,"gru=1.0,tr=1.0",6,True,True
7,d217626fd507,gru_transformer,0.762069,0.520980,6,6,6,3,2,285453,0.285453,96,2.0,1.0,4.0,2.0,"gru=2.0,tr=1.0",6,True,True


In [33]:
# Split into 4 DataFrames by model_name

MODEL_TYPES = ["gru", "gru_transformer", "transformer", "mamba"]

df_by_model = {}

for model_name in MODEL_TYPES:
    df_by_model[model_name] = rank_df[rank_df["model_name"] == model_name].copy()
    print(f"{model_name}: {len(df_by_model[model_name])} configs")

gru: 0 configs
gru_transformer: 8 configs
transformer: 0 configs
mamba: 0 configs


In [34]:
# Define ranking function per model

def rank_per_model(df: pd.DataFrame):
    df = df.copy()

    df["ctd_rank"] = df["avg_weighted_ctd"].rank(
        method="min", ascending=False, na_option="bottom"
    )

    df["prauc_rank"] = df["avg_toxic_pr_auc"].rank(
        method="min", ascending=False, na_option="bottom"
    )

    df["combined_rank_score"] = df["ctd_rank"] + df["prauc_rank"]

    df_ctd = df.sort_values(
        by=["avg_weighted_ctd", "avg_toxic_pr_auc"],
        ascending=[False, False],
    ).reset_index(drop=True)

    df_prauc = df.sort_values(
        by=["avg_toxic_pr_auc", "avg_weighted_ctd"],
        ascending=[False, False],
    ).reset_index(drop=True)

    df_combined = df.sort_values(
        by=["combined_rank_score", "ctd_rank", "prauc_rank"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    return df_ctd, df_prauc, df_combined

In [35]:
# Display top 5 per model for each ranking method

display_cols = [
    "config_id",
    "avg_weighted_ctd",
    "avg_toxic_pr_auc",
    "ctd_rank",
    "prauc_rank",
    "combined_rank_score",
    "n_ctd",
    "n_prauc",
]

optional_cols = [
    "trainable_params",
    "hidden_size",
    "depth_repr",
]

display_cols += [c for c in optional_cols if c in rank_df.columns]

for model_name, df_model in df_by_model.items():
    if df_model.empty:
        continue

    print(f"\n===== {model_name.upper()} =====")

    display_cols_model = display_cols.copy()
    if 'gru' in model_name:
        display_cols_model += [
            "gru_layers",
        ]
    if 'mamba' in model_name:
        display_cols_model += [
            "mamba_layers",
            "mamba_d_state",
            "mamba_d_conv",
            "mamba_expand",
        ]
    if 'transformer' in model_name:
        display_cols_model += [
            "transformer_layers",
            "transformer_heads",
            "transformer_ff_ratio",
        ]

    df_ctd, df_prauc, df_combined = rank_per_model(df_model)

    print("\nTop 5 by avg weighted CTD")
    display(df_ctd[display_cols_model].head(5))

    print("\nTop 5 by avg toxic PR AUC")
    display(df_prauc[display_cols_model].head(5))

    print("\nTop 5 by combined rank score")
    display(df_combined[display_cols_model].head(5))


===== GRU_TRANSFORMER =====

Top 5 by avg weighted CTD


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,gru_layers,transformer_layers,transformer_heads,transformer_ff_ratio
0,ab7ab945565f,0.774783,0.511980,1.0,2.0,3.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,2.0,3.0
1,3faa61ed2ce7,0.773341,0.509871,2.0,4.0,6.0,6,6,266637,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,4.0
2,9a1d0a1f723a,0.764895,0.478682,3.0,8.0,11.0,6,6,397293,96,"gru=2.0,tr=2.0",2.0,2.0,4.0,3.0
3,55bdbf1bed26,0.764878,0.506592,4.0,6.0,10.0,6,6,266637,96,"gru=1.0,tr=1.0",1.0,1.0,2.0,4.0
4,c2b98c4f4e50,0.763838,0.509056,5.0,5.0,10.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,3.0



Top 5 by avg toxic PR AUC


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,gru_layers,transformer_layers,transformer_heads,transformer_ff_ratio
0,d217626fd507,0.762069,0.520980,7.0,1.0,8.0,6,6,285453,96,"gru=2.0,tr=1.0",2.0,1.0,4.0,2.0
1,ab7ab945565f,0.774783,0.511980,1.0,2.0,3.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,2.0,3.0
2,bcc3408502b0,0.751563,0.511499,8.0,3.0,11.0,6,6,229581,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,2.0
3,3faa61ed2ce7,0.773341,0.509871,2.0,4.0,6.0,6,6,266637,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,4.0
4,c2b98c4f4e50,0.763838,0.509056,5.0,5.0,10.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,3.0



Top 5 by combined rank score


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,gru_layers,transformer_layers,transformer_heads,transformer_ff_ratio
0,ab7ab945565f,0.774783,0.511980,1.0,2.0,3.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,2.0,3.0
1,3faa61ed2ce7,0.773341,0.509871,2.0,4.0,6.0,6,6,266637,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,4.0
2,d217626fd507,0.762069,0.520980,7.0,1.0,8.0,6,6,285453,96,"gru=2.0,tr=1.0",2.0,1.0,4.0,2.0
3,55bdbf1bed26,0.764878,0.506592,4.0,6.0,10.0,6,6,266637,96,"gru=1.0,tr=1.0",1.0,1.0,2.0,4.0
4,c2b98c4f4e50,0.763838,0.509056,5.0,5.0,10.0,6,6,248109,96,"gru=1.0,tr=1.0",1.0,1.0,4.0,3.0
